# Macroeconomic Indicators and Market Data Extraction

This section outlines the process of extracting critical macroeconomic indicators and financial market data required for analyzing the interplay between the US economy and its financial markets.

The data is sourced from two authoritative platforms:
1. **Yahoo Finance (`yfinance`)**: Used to extract daily historical closing prices for the S&P 500 index (`^GSPC`) as a proxy for the broader equity market, the US Dollar Index (`DX-Y.NYB`) to gauge currency strength, and the 10-Year (`^TNX`) and 2-Year (`^IRX`) US Treasury Yields to monitor interest rate environments and yield curve dynamics.
2. **Federal Reserve Economic Data (FRED)**: Accessed via `pandas_datareader` to obtain fundamental macroeconomic signals, specifically the Gross Domestic Product (GDP) and the Consumer Price Index (CPI), which indicate overall economic growth and inflation trends, respectively.

These specific indicators were selected to form a comprehensive view of the US economic health and market sentiment from January 2000 to the present date.


In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime

# Define the extraction period: January 1, 2000 to the current date
start_date = '2000-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')

# ---------------------------------------------------------
# 1. Financial Market Data Extraction via Yahoo Finance
# ---------------------------------------------------------
# Define the trading tickers:
# ^GSPC: S&P 500 Index
# DX-Y.NYB: US Dollar Index
# ^TNX: 10-Year US Treasury Note Yield
# ^IRX: 13-Week (approx. 2-Year context) Treasury Bill Yield
# Note: For strict 2-Year, 'ZT=F' (futures) or ^IRX (13-week) are common,
# but we will use the standard tickers. Often, the 2-year yield is approximated or we can fetch FRED's DGS2.
# Here we fetch the requested Yahoo Finance tickers.
tickers = ['^GSPC', 'DX-Y.NYB', '^TNX', '^IRX']

# Download daily historical 'Close' prices
market_data = yf.download(tickers, start=start_date, end=end_date)['Close']

print("Market Data Shape:", market_data.shape)
print("\nMarket Data Head:")
print(market_data.head())

# ---------------------------------------------------------
# 2. Macroeconomic Data Extraction via FRED
# ---------------------------------------------------------
# Define the FRED series IDs:
# GDP: Gross Domestic Product
# CPIAUCSL: Consumer Price Index for All Urban Consumers: All Items
fred_series = ['GDP', 'CPIAUCSL']

# Fetch the data reading directly from FRED's CSV export endpoints,
# to avoid pandas_datareader incompatibilities with pandas 3.0+.
macro_data_frames = []
for series in fred_series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
    df = pd.read_csv(url, index_col='observation_date', parse_dates=True)
    df.index.name = 'Date'
    df.columns = [series]
    # Filter by date range
    df = df.loc[start_date:end_date]
    macro_data_frames.append(df)

# Concatenate the macroeconomic series along columns
macro_data = pd.concat(macro_data_frames, axis=1)

print("\nMacroeconomic Data Shape:", macro_data.shape)
print("\nMacroeconomic Data Head:")
print(macro_data.head())


# Data Tidying and Cleaning

In this phase of the analysis, we align and consolidate datasets spanning different temporal frequencies, aligning with Data Science best practices for time-series modeling.

**Handling Heterogeneous Frequencies via Forward-Filling:**
Financial market data operates on a daily frequency (trading days), whereas macroeconomic indicators are reported less frequently—monthly for the Consumer Price Index (CPI) and quarterly for Gross Domestic Product (GDP). In macroeconomic analysis, a reported indicator is considered the prevailing economic condition until the subsequent data release. Therefore, applying a forward-fill (`ffill()`) operation to the `macro_data` is the most logically and statistically sound approach to propagate these values across our daily timeline without introducing look-ahead bias.

**Data Consolidation and Cleaning:**
We will merge the `market_data` and the forward-filled `macro_data` using a left join based on their Datetime indices. This prioritizes the trading calendar. Finally, we will drop any remaining `NaN` values (e.g., initial dates before the first macroeconomic data point is available or anomalous gaps) using `dropna()`, resulting in a complete, tidy dataset ready for rigorous statistical modeling and exploratory data analysis.


In [5]:
# 1. Forward-fill the lower-frequency macroeconomic data
macro_data_ffill = macro_data.ffill()

# 2. Merge daily market data with the forward-filled macro data using a left join
df_merged = market_data.join(macro_data_ffill, how='left')

# 3. Clean the merged dataset by dropping remaining NaN values
df_merged = df_merged.dropna()

# 4. Verify data integrity and types
print("Merged Dataset Info:")
df_merged.info()

print("\nMerged Dataset Head:")
print(df_merged.head())


Merged Dataset Info:
<class 'pandas.DataFrame'>
DatetimeIndex: 200 entries, 2000-02-01 to 2025-12-01
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   DX-Y.NYB  200 non-null    float64
 1   ^GSPC     200 non-null    float64
 2   ^IRX      200 non-null    float64
 3   ^TNX      200 non-null    float64
 4   GDP       200 non-null    float64
 5   CPIAUCSL  200 non-null    float64
dtypes: float64(6)
memory usage: 10.9 KB

Merged Dataset Head:
              DX-Y.NYB        ^GSPC  ^IRX   ^TNX        GDP  CPIAUCSL
Date                                                                 
2000-02-01  104.919998  1409.280029  5.55  6.617  10002.179     170.0
2000-03-01  104.889999  1379.189941  5.60  6.379  10002.179     171.0
2000-05-01  109.629997  1468.250000  5.67  6.255  10247.720     171.2
2000-06-01  109.050003  1448.810059  5.56  6.195  10247.720     172.2
2000-08-01  110.050003  1438.099976  6.06  5.991  10318.165     172.7
